# Phase 1: Supervised Predictors

This notebook covers:
1. **Categorical Feature Encoding**: Label Encoding (ordinal) & One-Hot Encoding (nominal)
2. **Regression Suite**: Linear Regression, Polynomial Regression, Ridge ($L_2$), Lasso ($L_1$)
3. **Classification Suite**: Logistic Regression, KNN (Euclidean), Support Vector Machines (Linear, Poly, RBF)
4. **Decision Tree Architecture**: Gini Index vs. Entropy Splitting & Pre-Pruning

In [ ]:
from mlo.data.generator import generate_customer_dataset
from mlo.data.encoder import CategoricalEncoder
from mlo.data.preprocessor import DataPreprocessor
from mlo.supervised.regression import train_regression_suite
from mlo.supervised.classification import train_classification_suite
from mlo.supervised.trees import compare_tree_architectures
import pandas as pd

## 1. Data Generation & Feature Encoding

In [ ]:
df = generate_customer_dataset(n_samples=500, random_state=42)
encoder = CategoricalEncoder()
df_encoded = encoder.fit_transform(df)
print("Encoded Features:", encoder.feature_columns_)
df_encoded.head()

## 2. Continuous Metric Forecasting: LTV Regression & Regularization

In [ ]:
preproc = DataPreprocessor(scaler_type="standard")
X_train_r, X_test_r, y_train_r, y_test_r = preproc.split(
    df_encoded[encoder.feature_columns_], df_encoded['LTV'], test_size=0.2, random_state=42
)
reg_res = train_regression_suite(X_train_r.values, y_train_r.values, X_test_r.values, y_test_r.values)
for name, stats in reg_res.items():
    print(f"{name:25s} | MSE: {stats['mse']:10.2f} | R2: {stats['r2']:.4f}")

## 3. Churn Classification Suite (Logistic, KNN, SVM Kernels)

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = preproc.split(
    df_encoded[encoder.feature_columns_], df_encoded['Churn'], test_size=0.2, random_state=42, stratify=df_encoded['Churn']
)
X_tr_c_sc = preproc.fit_transform(X_train_c.values)
X_te_c_sc = preproc.transform(X_test_c.values)

suite, cls_df = train_classification_suite(X_tr_c_sc, y_train_c.values, X_te_c_sc, y_test_c.values)
cls_df

## 4. Decision Tree Splitting (Gini vs. Entropy) & Pre-Pruning

In [ ]:
tree_results = compare_tree_architectures(
    X_tr_c_sc, y_train_c.values, X_te_c_sc, y_test_c.values, feature_names=encoder.feature_columns_
)
print(f"Unpruned Gini Tree Test Accuracy: {tree_results['unpruned_gini']['test_acc']:.4f} (Depth: {tree_results['unpruned_gini']['depth']})")
print(f"Pre-pruned Entropy Tree Test Accuracy: {tree_results['pre_pruned_entropy']['test_acc']:.4f} (Depth: {tree_results['pre_pruned_entropy']['depth']})")
print("\n--- Extracted Rules ---\n", tree_results['pre_pruned_entropy']['rules'])